In [2]:
from langchain_community.document_loaders import TextLoader
from dotenv import load_dotenv

load_dotenv()

loader = TextLoader("data/news.txt", encoding="utf-8")
docs = loader.load()
print(f"총 글자수: {len(docs[0].page_content)}")
print("\n========앞부분 미리보기=========\n")
print(docs[0].page_content[:500])

총 글자수: 1708

========앞부분 미리보기=========

제목: 
AI2, 상업 활용까지 자유로운 '진짜' 오픈 소스 LLM '올모' 출시

내용:
앨런AI연구소(AI2)가 완전한 오픈 소스 대형언어모델(LLM) '올모(OLMo)’를 출시했다. 데이터 수집, 학습, 배포의 전 과정을 투명하게 공개한 데다 상업적 사용까지 허용한 진정한 의미의 오픈 소스 LLM이라는 평가다.
벤처비트는 1일(현지시간) 비영리 민간 AI 연구기관인 AI2가 ‘최초의 진정한 오픈 소스 LLM 및 프레임워크’라고 소개한 ‘올모’를 출시했다고 보도했다. 
이에 따르면 올모는 모델 코드와 모델 가중치뿐만 아니라 훈련 코드, 훈련 데이터, 관련 툴킷 및 평가 툴킷도 제공한다. 이를 통해 모델이 어떻게 구축되었는지 심층적으로 분석, LLM의 작동 방식과 응답을 생성하는 원리를 더 잘 이해할 수 있다. 
올모 프레임워크는 70억 매개변수의 ‘올모 7B’ 등 4가지 변형 모델과 10억 매개변수의 ‘올모 1B’ 모델을 제공한다. 모델들은 훈련 데이터를 생성하는 코드를 포함해 


In [3]:
from langsmith import Client

prompt = Client().pull_prompt(
    "teddynote/summary-stuff-documents-korean",
    dangerously_pull_public_prompt=True,
)
prompt.pretty_print()


Please summarize the sentence according to the following REQUEST.
REQUEST:
1. Summarize the main points in bullet points in KOREAN.
2. Each summarized sentence must start with an emoji that fits the meaning of the each sentence.
3. Use various emojis to make the summary more interesting.
4. Translate the summary into KOREAN if it is written in ENGLISH.
5. DO NOT translate any technical terms.
6. DO NOT include any unnecessary information.

CONTEXT:
{context}

SUMMARY:"



In [4]:
from langchain_openai import ChatOpenAI
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_teddynote.callbacks import StreamingCallback

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    streaming=True,
    callbacks=[StreamingCallback()],
)

stuff_chain = create_stuff_documents_chain(llm, prompt)
answer = stuff_chain.invoke({"context": docs})

- 🚀 AI2가 데이터 수집·학습·배포 전 과정을 공개하고 상업적 사용을 허용한 완전한 오픈 소스 LLM ‘OLMo’를 출시했다.
- 🔍 OLMo는 모델 코드·가중치뿐 아니라 훈련 코드·데이터·툴킷·평가 툴킷까지 제공해 LLM의 작동 원리를 분석할 수 있다.
- 🧩 1B·7B 등 다양한 모델 변형을 제공하며, 3조 개 토큰 규모의 Dolma 데이터 세트를 기반으로 구축됐다.
- 📊 추론 코드, 훈련 지표·로그, 전체 모델 가중치와 대규모 체크포인트 평가 제품군도 포함한다.
- ⚖️ Apache 2.0 라이선스를 적용해 상업적 활용에 제한이 없다.
- 🏆 일부 자연어 처리 벤치마크에서 Meta의 Llama와 TII의 Falcon 등 상업용 모델과 동등하거나 뛰어난 성능을 보였다.
- ⚠️ 비영어권 언어 품질과 코드 생성 기능은 상대적으로 부족하다.
- 🔬 AI2는 OLMo를 통해 투명하고 신뢰할 수 있는 LLM 연구 및 차세대 시스템 개발을 지원할 계획이다.
- 💾 OLMo의 모든 리소스는 GitHub와 Hugging Face에서 무료로 이용할 수 있다.

In [5]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """ Please summarize the sentence according to the following REQUEST.
    REQUEST:
    ...
    CONTEXT:
    {context}

    SUMMARY:
    """
)

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()
docs = docs[3:8]
print(f"총 페이지수: {len(docs)}")

총 페이지수: 5


In [9]:
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(temperature=0, model="gpt-5.6-luna",)

client = Client()
map_prompt = client.pull_prompt("teddynote/map-prompt", dangerously_pull_public_prompt=True)

map_prompt.pretty_print()

================================ System Message ================================

You are a professional main thesis extractor.

================================ Human Message =================================

Your task is to extract main thesis from given documents. Answer should be in same language as given document. 

#Format: 
- thesis 1
- thesis 2
- thesis 3
- ...

Here is a given document: 
{doc}

Write 1~5 sentences.
#Answer:


In [10]:
map_chain = map_prompt | llm | StrOutputParser()

In [11]:
doc_summaries = map_chain.batch(docs)

In [12]:
print(doc_summaries[0])

- 미국 바이든 대통령은 2023년 10월 30일 안전하고 신뢰할 수 있는 AI의 개발과 사용을 보장하기 위한 행정명령을 발표했다.
- 행정명령은 AI 안전·보안 기준 마련, 개인정보보호, 형평성과 시민권 증진, 소비자 보호, 노동자 지원, 혁신·경쟁 촉진 및 국제협력을 포괄한다.
- 특히 고성능 AI 개발 기업에 안전성 테스트 결과와 주요 정보를 정부에 공유하도록 요구하고, AI 생성 콘텐츠 표시 기준과 안전성 검증 표준 마련을 추진한다.
- 미국은 AI로 인한 차별과 노동자 피해를 방지하는 동시에 국가 AI 연구 인프라 확충, 중소기업 지원, 해외 전문인력 유치 등을 통해 AI 혁신과 경쟁력을 강화하고자 한다.


In [14]:
# reduce prompt 다운로드
reduce_prompt = client.pull_prompt("teddynote/reduce-prompt", dangerously_pull_public_prompt=True)

# 프롬프트 출력
reduce_prompt.pretty_print()

================================ System Message ================================

You are a professional summarizer. You are given a list of summaries of documents and you are asked to create a single summary of the documents.

================================ Human Message =================================

#Instructions: 
1. Extract main points from a list of summaries of documents
2. Make final summaries in bullet points format.
3. Answer should be written in {language}.

#Format: 
- summary 1
- summary 2
- summary 3
- ...

Here is a list of summaries of documents: 
{doc_summaries}

#SUMMARY:


In [15]:
reduce_chain = reduce_prompt | llm | StrOutputParser()

In [16]:
from langchain_teddynote.messages import stream_response

answer = reduce_chain.stream(
    {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
)

stream_response(answer)

- 미국 바이든 행정부는 2023년 10월 행정명령을 통해 AI 안전·보안 기준, 개인정보보호, 차별 방지, 소비자·노동자 보호, 혁신 촉진 및 국제협력을 포괄하는 AI 규제·진흥 정책을 추진했다.
- 고성능 AI 개발 기업에 안전성 테스트 결과와 주요 정보를 정부에 공유하도록 요구하고, AI 생성 콘텐츠 표시·출처 확인 및 안전성 검증 표준 마련을 추진하는 등 투명성과 책임성을 강화하고자 했다.
- G7 히로시마 AI 프로세스와 28개국 블레츨리 선언은 첨단 AI의 위험을 국제적으로 관리하기 위해 AI 수명주기 전반의 위험 평가, 투명성, 책임성, 보안 통제, 정보 공유 및 공동 표준 개발을 강조했다.
- 각국 정부·기업·학계·시민사회는 AI 안전성 테스트와 연구개발에 협력하기로 했으며, 영국 AI 안전 연구소가 국제적 안전성 평가를 주도하고 한국은 후속 AI 미니 정상회의를 공동 개최하기로 했다.
- 미국 법원은 생성 AI 이미지와 특정 저작물 간 실질적 유사성 및 저작권 침해 증거가 부족하다는 이유로 예술가들의 소송 일부를 기각했지만, 특정 작품의 무단 복제와 관련된 소송은 계속 진행될 수 있다고 판단했다.
- 미국 FTC는 생성 AI가 개인정보 침해, 차별·편견, 사기, 창작자 권리 침해와 시장 독점 심화 등의 위험을 초래할 수 있다고 지적하며, 현행 법 집행 권한을 활용해 소비자 보호와 공정한 경쟁을 확보하겠다고 밝혔다.
- 전반적으로 미국과 국제사회는 AI 혁신을 촉진하는 동시에 안전성, 인권, 저작권, 개인정보 및 시장 경쟁을 보호하기 위한 규범과 협력 체계를 구축하고 있다.

In [19]:
from langchain_core.runnables import chain

@chain
def map_reduce_chain(docs):
    map_llm = ChatOpenAI(
        temperature=0, model="gpt-5.6-luna",
    )

    # map_prompt 다운로드
    map_prompt = client.pull_prompt("teddynote/map-prompt", dangerously_pull_public_prompt=True)

    # map chain 생성
    map_chain = map_prompt | map_llm | StrOutputParser()

    # 첫 번째 프롬프트, ChatOpenAI, 문자열 출력 파서를 연결하여 체인을 생성
    doc_summaries = map_chain.batch(docs)

    # reduce_prompt 다운로드
    reduce_prompt = client.pull_prompt("teddynote/reduce-prompt", dangerously_pull_public_prompt=True)
    reduce_llm = ChatOpenAI(
        model="gpt-5.6-luna",
        callbacks=[StreamingCallback()],
        streaming=True,
    )

    reduce_chain = reduce_prompt | reduce_llm | StrOutputParser()

    return reduce_chain.invoke(
        {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
    )

In [20]:
answer = map_reduce_chain.invoke(docs)

- 미국 바이든 행정부는 안전하고 신뢰할 수 있는 AI를 위해 안전·보안 기준, 개인정보 보호, 형평성·시민권, 소비자·노동자 보호, 혁신·경쟁, 국제협력을 아우르는 종합 행정명령을 발표했다.
- 고성능 AI 개발 기업에 안전성 평가 결과와 주요 정보를 정부에 제출하도록 요구하고, AI 생성 콘텐츠의 표시·출처 인증과 안전성·신뢰성 검증 체계 마련을 추진한다.
- 법률·주택·보건·형사사법·교육·의료 등 분야에서 AI로 인한 차별과 편견을 방지하고, 근로자 보호와 책임 있는 AI 활용을 강화한다.
- 미국은 국가 AI 연구 인프라 확대, 중소기업·개발자 지원, 해외 전문인력 유입을 통해 AI 혁신과 국가 경쟁력을 높일 계획이다.
- G7 히로시마 AI 프로세스는 첨단 AI 기업이 개발부터 배포까지 전 생애주기의 위험을 식별·완화하도록 하는 자발적 국제 행동강령을 마련했다.
- 국제 행동강령은 투명성·책임성, 정보 공유, 이해관계자 협력, 보안 통제, 개인정보·지식재산권 보호, 콘텐츠 워터마크와 출처 확인, 국제 표준 개발 등을 강조한다.
- 영국 AI 안전성 정상회의 참가 28개국은 블레츨리 선언을 통해 정부·국제기구·기업·시민사회·학계가 첨단 AI의 위험에 공동 대응하기로 합의했다.
- 각국은 기업의 안전 평가와 투명성 강화 책임을 확인하고, 정부 주도 외부 안전 테스트, 테스트 결과 공유, 공동 표준과 AI 안전 연구를 추진하기로 했다.
- 영국 AI 안전 연구소는 첨단 AI 모델의 외부 안전 테스트를 주도하며, 참가국들은 첨단 AI의 위험과 가능성을 평가하는 ‘과학의 현황’ 보고서 작성에도 합의했다.
- 미국 법원은 생성 AI 저작권 소송 일부를 저작권 미등록과 증거 불충분을 이유로 기각했으며, 생성 이미지와 특정 원작 간 실질적 유사성을 입증해야 침해를 인정할 수 있다고 판단했다.
- 다만 스태빌리티AI가 특정 작가의 저작물 16점을 무단 복제했다는 혐의는 계속 심리할 수 있도록 허용해, 원고들이 침해 이미지 중심으로 소송을 수정해 재제기할 수 있게 했다